In [ ]:
from datetime import date

import numpy as np
import pandas as pd
import yaml
from sqlalchemy import create_engine
##preguntas que responde este hecho
######1.2.4.6.

##origen servicio como sede

##usuario pertenece a sede

##usuario pertenece a solo una sede

# database connections 

In [ ]:
with open('../config_fill.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['CO_SA']
    config_etl = config['ETL_PRO']

url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
co_sa = create_engine(url_co)
etl_conn = create_engine(url_etl)

# Extract

In [ ]:
#hecho_servicio = pd.read_sql_table('mensajeria_servicio', url_co)
tabla_fases = pd.read_sql_table('mensajeria_estadosservicio', url_co)
dim_cliente = pd.read_sql_table('dim_cliente', etl_conn)
dim_sede = pd.read_sql_table('dim_sede', etl_conn)
usuarios = pd.read_sql_table('clientes_usuarioaquitoy', co_sa)


In [ ]:
dim_sede=dim_sede[['sede_id','cliente_id']]

# Transformations



In [ ]:
hecho_servicio = pd.read_sql_table('mensajeria_servicio', url_co)

In [ ]:
hecho_servicio.drop(columns=['descripcion','fecha_deseada','hora_deseada', 'nombre_recibe','telefono_recibe','descripcion_pago','ida_y_regreso','activo','novedades','tipo_pago_id','tipo_vehiculo_id','prioridad','hora_visto_por_mensajero','visto_por_mensajero','descripcion_multiples_origenes','mensajero2_id','mensajero3_id','multiples_origenes','asignar_mensajero','es_prueba','descripcion_cancelado'], inplace=True)

hecho_servicio.drop(columns=['destino_id','mensajero_id','origen_id','tipo_servicio_id','ciudad_destino_id','ciudad_origen_id'], inplace=True)

hecho_servicio.columns.tolist()


In [ ]:
hecho_servicio.info()

# Alinear nombres de columnas al diagrama
ID_Estado (PK) -> indice al cargar | Nombre_Estado | Orden_Secuencia

In [ ]:
hecho_servicio.rename(columns={
    'id': 'id_servicio','cliente_id': 'id_cliente'
}, inplace=True)

In [ ]:
usuarios_min = usuarios[['id', 'sede_id']]
hecho_servicio = hecho_servicio.merge(usuarios_min, left_on='usuario_id', right_on='id', how='left')

hecho_servicio.drop(columns=['usuario_id', 'id'], inplace=True)

hecho_servicio.rename(columns={
    'sede_id': 'id_sede'
}, inplace=True)


In [ ]:
#hecho_servicio.head()
#hecho_servicio['id_cliente'].unique()
hecho_servicio.info()

# load

In [ ]:

hecho_servicio.to_sql('hecho_servicio', etl_conn, if_exists='replace', index_label='key_hecho_servicio')